In [ ]:
file = '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/NGC5194_kpno09m_HAedit.fits'
hdu = fits.open(file)[0]
hdu.header

In [ ]:
#This cell will load the image objects for all four galaxies
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table.add_column([0.0]*len(table), name = 'pa_EW')
table.add_column([0.0]*len(table), name = 'ha_EW')
table.add_column([0.0]*len(table), name = 'ha_EW_from_image')
table.add_column([0.0]*len(table), name = 'pa_EW_from_image')
ngc1672 = ImageScience()
ngc1672.load_object('/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1672_image_science.pkl')
ngc1512 = ImageScience()
ngc1512.load_object('/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1512_image_science.pkl')
ngc1433 = ImageScience()
ngc1433.load_object('/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1433_image_science.pkl')
m51 = ImageScience()
m51.load_object('/project/galaxies/tjuchau/data_files/M51/Custom_objects/m51_image_science.pkl')
m51.load_image('ha_full', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/NGC5194_kpno09m_HAedit.fits')
m51.load_image('ha_EW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/NGC5194_kpno09m_HAEW.fits')
m51.load_image('F658N', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_NGC5194_ACS_WFC_drc_1.fits')
m51.load_image('ha_sings', '/project/galaxies/tjuchau/data_files/HST/ngc5194/ngc5194_sings_ha_correct.fits')
m51.display(['ha_full', 'ha_EW', 'F658N', 'ha_sings'], loc_sky, radius, background_annulus_thickness=0.1*u.arcsec, buffer=0.1*u.arcsec, ncols=4, cmap='viridis', zoom=5, show_grid=False)


In [ ]:
#This cell recreates the image objects for the four galaxies, this takes about 40 mins
#Objects are saved at the end, likely do not need to rerun this cell ever

ngc1672 = ImageScience()
output_file = '/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1672_image_science.pkl'
files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1672.load_image(names[i], file)
ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1672.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1672.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1672.images.keys()}')

ngc1672.align_images('pa_cont', 'ha_cont')
ngc1672.align_images('pa_contsub', 'ha_contsub')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)
ngc1672.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posEWonly.fits')
ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)
ngc1672.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posHaEWonly.fits')
ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)
ngc1672.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_finiteEWonly.fits')
ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)
ngc1672.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posPaEWonly.fits')
ngc1672.make_ratio('pa_contsub', 'pa_cont', out_name='pa_EW', out_file=None, scales=[1,1])
ngc1672.save_object(output_file)
################################################################################
#NGC1512
################################################################################
ngc1512 = ImageScience()
output_file =  '/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1512_image_science.pkl'
files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1512.load_image(names[i], file)
ngc1512.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1512_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1512.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1512_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1512.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1512.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1512.images.keys()}')

ngc1512.align_images('pa_cont', 'ha_cont')
ngc1512.align_images('pa_contsub', 'ha_contsub')

ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)
ngc1512.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posEWonly.fits')
ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)
ngc1512.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posHaEWonly.fits')
ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)
ngc1512.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_finiteEWonly.fits')
ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)
ngc1512.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posPaEWonly.fits')
ngc1512.make_ratio('pa_contsub', 'pa_cont', out_name='pa_EW', out_file=None, scales=[1,1])
ngc1512.save_object(output_file)
################################################################################
#NGC1433
################################################################################

ngc1433 = ImageScience()
output_file = '/project/galaxies/tjuchau/data_files/M51/Custom_objects/ngc1433_image_science.pkl'
files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1433.load_image(names[i], file)
ngc1433.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1433.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1433.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1433.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1512.images.keys()}')

ngc1433.align_images('pa_cont', 'ha_cont')
ngc1433.align_images('pa_contsub', 'ha_contsub')

ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)
ngc1433.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posEWonly.fits')
ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)
ngc1433.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posHaEWonly.fits')
ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)
ngc1433.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_finiteEWonly.fits')
ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)
ngc1433.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posPaEWonly.fits')
ngc1433.make_ratio('pa_contsub', 'pa_cont', out_name='pa_EW', out_file=None, scales=[1,1])
ngc1433.save_object(output_file)

################################################################################
#M51 (not implemented yet)
################################################################################

m51 = ImageScience()
output_file = '/project/galaxies/tjuchau/data_files/M51/Custom_objects/m51_image_science.pkl'
#m51.load_image('ha_full', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/NGC5194_kpno09m_HAedit.fits')
#m51.load_image('ha_EW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/NGC5194_kpno09m_HAEW.fits')
#m51.load_image('F658N', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_NGC5194_ACS_WFC_drc_1.fits')
#m51.load_image('ha_sings', '/project/galaxies/tjuchau/data_files/HST/ngc5194/ngc5194_sings_ha_correct.fits')
'''raw_image_files = glob.glob('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/*anchor.fits')
danny_files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc5194/*')

m51.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
m51.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
m51.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
m51.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1512.images.keys()}')

m51.align_images('pa_cont', 'ha_cont')
m51.align_images('pa_contsub', 'ha_contsub')

m51.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)
m51.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posEWonly.fits')
m51.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)
m51.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posHaEWonly.fits')
m51.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)
m51.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_finiteEWonly.fits')
m51.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)
m51.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posPaEWonly.fits')
'''
m51.save_object(output_file)

for row in table:
    if row['galaxy'] == 'M51':
        obj = m51
        continue
    elif row['galaxy'] == 'ngc1433':
        obj = ngc1433
    elif row['galaxy'] == 'ngc1672':
        obj = ngc1672
    elif row['galaxy'] == 'ngc1512':
        obj = ngc1512
    numerator = obj.get_background_subtracted_flux('pa_contsub', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec, buffer=0.1*u.arcsec)['net_flux']
    denominator = obj.get_background_subtracted_flux('pa_cont', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec, buffer=0.1*u.arcsec)['net_flux']
    pa_EW = numerator / denominator
    if np.isfinite(pa_EW):
        row['pa_EW'] = pa_EW
    numerator = obj.get_background_subtracted_flux('ha_contsub', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec, buffer=0.1*u.arcsec)['net_flux']
    denominator = obj.get_background_subtracted_flux('ha_cont', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec, buffer=0.1*u.arcsec)['net_flux']
    ha_EW = numerator / denominator
    if np.isfinite(ha_EW):
        row['ha_EW'] = ha_EW

plt.scatter(table[table['galaxy'] != 'M51']['best.stellar.age_m_star'], table[table['galaxy'] != 'M51']['pa_EW'])
plt.show()

In [ ]:
table[table['galaxy'] != 'M51']

In [ ]:
#This cell is only for ngc1672
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience
ngc1672 = ImageScience()

files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1672.load_image(names[i], file)
ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1672.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1672.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1672.images.keys()}')

table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table.add_column([0]*len(table), name = 'pa_EW')
table.add_column([0]*len(table), name = 'test1')
table.add_column([0]*len(table), name = 'test2')
table.add_column([0]*len(table), name = 'test3')
table.add_column([0]*len(table), name = 'ha_EW')
table = table[table['galaxy']=="ngc1672"]


ngc1672.align_images('pa_cont', 'ha_cont')
ngc1672.align_images('pa_contsub', 'ha_contsub')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)

ngc1672.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posEWonly.fits')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)

ngc1672.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posHaEWonly.fits')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)

ngc1672.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_finiteEWonly.fits')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)

ngc1672.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/ngc1672_PaaEW_over_HaEW_posPaEWonly.fits')


In [ ]:
print('both forced to be positive',len(ngc1672.images['EW_Paa_over_Ha_posEW'][ngc1672.images['EW_Paa_over_Ha_posEW'] <0]))
print('negatives after Ha forced to be positive',len(ngc1672.images['EW_Paa_over_Ha_posHa'][ngc1672.images['EW_Paa_over_Ha_posHa'] <0]))
print('negatives after Pa forced to be postive',len(ngc1672.images['EW_Paa_over_Ha_posPa'][ngc1672.images['EW_Paa_over_Ha_posPa'] <0]))
print('negatives after only finite values kept',len(ngc1672.images['EW_Paa_over_Ha'][ngc1672.images['EW_Paa_over_Ha'] <0]))

In [ ]:
#This cell is only for ngc1512
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience
ngc1512 = ImageScience()

files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1512.load_image(names[i], file)
ngc1512.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1512_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1512.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1512_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1512.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1512.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1512.images.keys()}')

table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table.add_column([0]*len(table), name = 'pa_EW')
table.add_column([0]*len(table), name = 'test1')
table.add_column([0]*len(table), name = 'test2')
table.add_column([0]*len(table), name = 'test3')
table.add_column([0]*len(table), name = 'ha_EW')
table = table[table['galaxy']=="ngc1512"]


ngc1512.align_images('pa_cont', 'ha_cont')
ngc1512.align_images('pa_contsub', 'ha_contsub')

ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)

ngc1512.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posEWonly.fits')

ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)

ngc1512.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posHaEWonly.fits')

ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)

ngc1512.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_finiteEWonly.fits')

ngc1512.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)

ngc1512.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1512/ngc1512_PaaEW_over_HaEW_posPaEWonly.fits')

print('both forced to be positive',len(ngc1512.images['EW_Paa_over_Ha_posEW'][ngc1512.images['EW_Paa_over_Ha_posEW'] <0]))
print('negatives after Ha forced to be positive',len(ngc1512.images['EW_Paa_over_Ha_posHa'][ngc1512.images['EW_Paa_over_Ha_posHa'] <0]))
print('negatives after Pa forced to be postive',len(ngc1512.images['EW_Paa_over_Ha_posPa'][ngc1512.images['EW_Paa_over_Ha_posPa'] <0]))
print('negatives after only finite values kept',len(ngc1512.images['EW_Paa_over_Ha'][ngc1512.images['EW_Paa_over_Ha'] <0]))

In [ ]:
#This cell is only for ngc1433
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience
ngc1433 = ImageScience()

files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
checks = ['continuum_flux_err', 'continuum_flux_log', 'contsub_flux_err', 'contsub_flux_log', 'halpha_flux_nii_corr_err', 'halpha_flux_nii_corr_log']
for i, file in enumerate(files):
    if checks[i] not in file:
        print('FILES HAVE CHANGED!!!')
        break
    ngc1433.load_image(names[i], file)
ngc1433.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1433.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1433_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1433.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1433.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1512.images.keys()}')

table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table.add_column([0]*len(table), name = 'pa_EW')
table.add_column([0]*len(table), name = 'test1')
table.add_column([0]*len(table), name = 'test2')
table.add_column([0]*len(table), name = 'test3')
table.add_column([0]*len(table), name = 'ha_EW')
table = table[table['galaxy']=="ngc1433"]


ngc1433.align_images('pa_cont', 'ha_cont')
ngc1433.align_images('pa_contsub', 'ha_contsub')

ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posEW', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)

ngc1433.save_fits('EW_Paa_over_Ha_posEW', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posEWonly.fits')

ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posHa', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = True)

ngc1433.save_fits('EW_Paa_over_Ha_posHa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posHaEWonly.fits')

ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = False, replace_den_negs = False)

ngc1433.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_finiteEWonly.fits')

ngc1433.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha_posPa', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = False)

ngc1433.save_fits('EW_Paa_over_Ha_posPa', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1433/ngc1433_PaaEW_over_HaEW_posPaEWonly.fits')

print('both forced to be positive',len(ngc1433.images['EW_Paa_over_Ha_posEW'][ngc1433.images['EW_Paa_over_Ha_posEW'] <0]))
print('negatives after Ha forced to be positive',len(ngc1433.images['EW_Paa_over_Ha_posHa'][ngc1433.images['EW_Paa_over_Ha_posHa'] <0]))
print('negatives after Pa forced to be postive',len(ngc1433.images['EW_Paa_over_Ha_posPa'][ngc1433.images['EW_Paa_over_Ha_posPa'] <0]))
print('negatives after only finite values kept',len(ngc1433.images['EW_Paa_over_Ha'][ngc1433.images['EW_Paa_over_Ha'] <0]))

In [ ]:
ngc1672.display(['ha_cont', 'ha_contsub', 'pa_cont', 'pa_contsub'], [table[0]['ra'], table[0]['dec']], 0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer=0.1*u.arcsec, ncols=2)

In [ ]:
for row in table:
    ha_ew = ngc1672.get_equivalent_width('ha_full', 'ha_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
    ha_ew_nobg = ngc1672.get_equivalent_width('ha_full', 'ha_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0*u.arcsec, buffer=0*u.arcsec)
    pa_ew = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
    pa_ew_nobg = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0*u.arcsec, buffer=0*u.arcsec)
    row['ha_EW'] = ha_ew[0].value
    row['pa_EW'] = pa_ew[0].value
    row['test1'] = pa_ew_nobg[0].value
    row['test2'] = ha_ew_nobg[0].value


In [ ]:
plt.clf()
plt.scatter(table['test2'], table['test1'])
plt.xlabel('H-alpha')
plt.ylabel('Pa-alpha')
plt.ylim([np.min(table['pa_EW']), 6000])

plt.show()

In [ ]:
plt.clf()
plt.scatter(table['ha_EW'], table['pa_EW'])
plt.xlabel('H-alpha')
plt.ylabel('Pa-alpha')
plt.ylim([np.min(table['pa_EW']), 6000])

plt.show()

In [ ]:
pa_ew = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
cont_est = ngc1672.get_background_subtracted_flux('pa_cont', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
full_flux = ngc1672.get_background_subtracted_flux('pa_full', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)


In [ ]:
ngc1672.display(['pa_full', 'pa_cont', 'pa_contsub'], 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.3*u.arcsec,ncols=2)

In [ ]:
ngc1672.get_background_subtracted_flux('pa_full', 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.3*u.arcsec)

In [ ]:
ngc1672.get_background_subtracted_flux('pa_full', 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.1*u.arcsec)

In [ ]:
ngc1672.align_images('pa_cont', 'ha_cont')
ngc1672.align_images('pa_contsub', 'ha_contsub')

ngc1672.make_ew_ratio_image('pa_cont', 'pa_contsub', 'ha_cont_aligned', 'ha_contsub_aligned',
    output_name='EW_Paa_over_Ha', min_continuum=0, min_line=-np.inf, replace_num_negs = True, replace_den_negs = True)

ngc1672.save_fits('EW_Paa_over_Ha', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/EW_Paa_over_Ha_no_negatives.fits')



In [ ]:
from astropy.io import fits
import numpy as np

input_file = '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/EW_Paa_over_Ha.fits'
output_file = '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/EW_Paa_over_Ha_embedded.fits'

# Load the FITS file
with fits.open(input_file) as hdul:

    # Copy the data so the original isn't modified
    data = hdul[0].data.copy()

    # Replace negative values with zero
    data[data < 1] = 0

    # Put the modified data back into the HDU
    hdul[0].data = data

    # Save to a new file
    hdul.writeto(output_file, overwrite=True)

In [ ]:
fits.open('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/EW_Paa_over_Ha.fits').info()